In [ ]:
import timm
import os, math, random, logging
import numpy as np
from PIL import Image
from tqdm.auto import tqdm
import json

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from torchvision import transforms
from torchvision.transforms import functional as TF
import torchvision.utils as vutils
import matplotlib.pyplot as plt

from accelerate import Accelerator
from diffusers import DDPMScheduler, UNet2DConditionModel, AutoencoderKL
from diffusers import DDIMScheduler

In [ ]:
"This cell is for building the Diffusion Decoder's dataset"

import os
import json
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.cuda.amp import autocast
from tqdm.auto import tqdm

from diffusers import AutoencoderKL


# ================================================================
# VAE Encoder Wrapper
# ================================================================
class VAEEncoder(nn.Module):
    """
    A lightweight wrapper that loads the Stable Diffusion VAE
    and exposes an encode() function that outputs latents.
    """

    def __init__(self, pretrained_path="runwayml/stable-diffusion-v1-5", device="cuda"):
        super().__init__()
        self.device = device

        print("[VAEEncoder] Loading VAE from:", pretrained_path)
        self.vae: AutoencoderKL = AutoencoderKL.from_pretrained(
            pretrained_path, subfolder="vae"
        ).to(device)

        self.vae.eval()
        for p in self.vae.parameters():
            p.requires_grad_(False)

        # Stable Diffusion VAE scaling factor
        self.scaling = self.vae.config.scaling_factor  # usually 0.18215

    @torch.no_grad()
    def encode(self, imgs):
        """
        imgs:  [B,3,512,512] in [-1,1]
        return: [B,4,64,64] latent tensor
        """
        latents = self.vae.encode(imgs).latent_dist.sample()
        latents = latents * self.scaling
        return latents


# ================================================================
# Main Precomputation Function (NO masks, NO diffusion)
# ================================================================
def precompute_latents(
    root_dir,
    out_dir,
    masks_per_image=100,
    vae_path="runwayml/stable-diffusion-v1-5",
    device="cuda",
    batch_size=16,
    num_workers=4,
    amp_dtype=torch.float16
):
    """
    Loads images from OutpaintDataset, applies the dataset's augmentations
    (random rotation/flip), encodes each image with the VAE into latent z,
    and saves:

        {
            "z_cond": [4,64,64] fp16,
            "target_img": [3,512,512] fp16,
            "src_path": "...",
            "k": int
        }

    No mask is generated and no diffusion model is used.
    """

    import torch
    from tqdm.auto import tqdm

    os.makedirs(out_dir, exist_ok=True)
    index_path = os.path.join(out_dir, "index.jsonl")

    # Dataset provides (augmented_img, original_img)
    ds = OutpaintDataset(
        root_dir=root_dir,
        img_size=512,
        masks_per_image=masks_per_image,  # still used to form (img, k)
        no_mask=True                       # you should ensure your dataset ignores mask logic
    )

    dl = DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
        drop_last=False
    )

    # Load VAE encoder only
    encoder = VAEEncoder(pretrained_path=vae_path, device=device)

    def toCPU16(x):
        return x.detach().cpu().half()

    global_idx = 0

    with open(index_path, "w") as fw:
        pbar = tqdm(dl, desc=f"Precomputing from {root_dir}")
        for batch in pbar:
            # Dataset returns: (augmented_img, target_img)
            aug_img, target_img = batch

            aug_b = aug_img.to(device, non_blocking=True)

            # Encode to latent using VAE
            with torch.no_grad():
                with autocast():
                    z_cond = encoder.encode(aug_b)   # [B,4,64,64]

            B = aug_b.size(0)

            for b in range(B):
                idx = global_idx + b
                img_idx = idx // ds.masks_per_image
                k      = idx %  ds.masks_per_image

                base = os.path.splitext(os.path.basename(ds.img_files[img_idx]))[0]
                pt_path = os.path.join(out_dir, f"{base}_k{k:03d}.pt")

                torch.save({
                    "z_cond":     toCPU16(z_cond[b]),
                    "target_img": toCPU16(target_img[b]),
                    "src_path":   ds.img_files[img_idx],
                    "k":          int(k)
                }, pt_path)

                fw.write(json.dumps({"pt": pt_path}) + "\n")

            global_idx += B

            if global_idx % (batch_size * 10) == 0:
                torch.cuda.empty_cache()

    print(f"[Precompute] Done. Index saved at: {index_path}")


In [ ]:
"This cell is for executing the dataset building"
precompute_latents(
    root_dir="origin_img_folder",
    out_dir="output_img_folder",
    masks_per_image=60,
    vae_path="runwayml/stable-diffusion-v1-5",
    device="cuda"
)

In [ ]:
"This cell is for visualizing/check the built dataset"
import os, torch
import torchvision.utils as vutils
from diffusers import AutoencoderKL

@torch.no_grad()
def visualize_saved_latent(pt_path, vae, out_path="latent_vis.png"):
    """
    Load a saved .pt latent file, decode the latent using the VAE,
    and save the reconstructed image.

    Args:
        pt_path (str): Path to the saved .pt latent file.
        vae (AutoencoderKL): Loaded Stable Diffusion VAE model.
        out_path (str): File path to save the decoded image.
    """
    # Load the PT package (contains z_cond, target_img, etc.)
    pack = torch.load(pt_path, map_location="cpu")

    # Extract latent and add batch dimension → [1, 4, 64, 64]
    z = pack["z_cond"].unsqueeze(0).float()

    # Move latent to same device as the VAE
    device = next(vae.parameters()).device
    z = z.to(device)

    # VAE decoding requires dividing by scaling_factor
    z = z / vae.config.scaling_factor

    # Decode latent → produce an image in [-1, 1]
    img = vae.decode(z).sample

    # Convert to [0,1] range for saving
    img = (img.clamp(-1, 1) + 1) / 2

    # Save image
    vutils.save_image(img, out_path, nrow=1)
    print(f"Saved visualization to {out_path}")


# ==== Example Usage ====

# Load Stable Diffusion VAE
vae = AutoencoderKL.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    subfolder="vae"
).cuda().eval()

# Decode one of your saved latent files
visualize_saved_latent(
    "latent_file",
    vae,
    "output_filename"
)
